In [1]:
import numpy as np
import pandas as pd
from src.utils import get_all_paths

In [2]:
weights_paths = get_all_paths("/Users/saru/Local_Work/neuromodul/fscv/tests_code/data_1d_vxlbl")
voltammograms_path = "/Users/saru/Local_Work/neuromodul/fscv/tests_code/data_1d_vxlbl/AFOR/bfa/155Vs/10Hz/2025_07_07__AFOR__DA_5HT_NE__155Vs_10Hz__BFA_INM001_99bW06R09/DA_5HT_NE/voltammograms.npy"
labels_path = "/Users/saru/Local_Work/neuromodul/fscv/tests_code/data_1d_vxlbl/AFOR/bfa/155Vs/10Hz/2025_07_07__AFOR__DA_5HT_NE__155Vs_10Hz__BFA_INM001_99bW06R09/DA_5HT_NE/labels.npy"


In [ ]:
from src.utils import load_activation_data
train, val, test, y_mean, y_std = load_activation_data(weights_paths, batch_size=32)


Expert activation set shape:  (65, 150, 1000, 80)
Labels(concentrations) shape:  (65, 150, 4)
Duplicated conditions:
  DA=0.0, 5HT=0.0, NE=0.0 — appears 4 times
  DA=0.0, 5HT=0.0, NE=500.0 — appears 2 times
  DA=0.0, 5HT=0.0, NE=1000.0 — appears 2 times
  DA=0.0, 5HT=0.0, NE=1500.0 — appears 2 times
  DA=0.0, 5HT=500.0, NE=0.0 — appears 2 times
  DA=0.0, 5HT=500.0, NE=500.0 — appears 2 times
  DA=0.0, 5HT=1000.0, NE=0.0 — appears 2 times
  DA=0.0, 5HT=1500.0, NE=0.0 — appears 2 times
  DA=500.0, 5HT=0.0, NE=0.0 — appears 2 times
  DA=500.0, 5HT=0.0, NE=500.0 — appears 2 times
  DA=500.0, 5HT=500.0, NE=0.0 — appears 2 times
  DA=500.0, 5HT=500.0, NE=500.0 — appears 2 times
  DA=1000.0, 5HT=0.0, NE=0.0 — appears 2 times
  DA=1000.0, 5HT=500.0, NE=1000.0 — appears 2 times
  DA=1500.0, 5HT=0.0, NE=0.0 — appears 2 times
y_train mean: [794.13043 192.17392 192.17392]; std:[768.2031 376.0992 376.1   ]


In [4]:
from src.reg_models import Regressor, train_regressor, test_regressor
from torch.optim import Adam
from torch import nn

In [5]:
loss = nn.MSELoss()

In [ ]:
losses = {}
modes = ['attention', 'mean', 'none']

for mode in modes:
    print("----- Mode: ", mode,"--------")
    model = Regressor(80, 3, pooling=mode)
    optim = Adam(model.parameters(), lr=1e-3)
    train_loss = train_regressor(model=model,train_data_loader=train, val_loader=val,loss=loss,optimizer=optim, device="mps", num_epochs=100)

    losses[mode+" train"] = train_loss
    test_loss = test_regressor(model=model,test_data_loader=test,loss=loss,y_mean=y_mean,y_std=y_std, device='mps')
    losses[mode+" test"] = test_loss


----- Mode:  attention --------
Epoch 10/100| Train Loss: 0.8006| Val Loss: 0.9875
Epoch 20/100| Train Loss: 0.7843| Val Loss: 1.0138
Epoch 30/100| Train Loss: 0.7754| Val Loss: 1.0354
Epoch 40/100| Train Loss: 0.7726| Val Loss: 1.0194
Epoch 50/100| Train Loss: 0.7707| Val Loss: 1.0155
Epoch 60/100| Train Loss: 0.7688| Val Loss: 1.0194
Epoch 70/100| Train Loss: 0.7674| Val Loss: 1.0202
Epoch 80/100| Train Loss: 0.7642| Val Loss: 1.0190
Epoch 90/100| Train Loss: 0.7665| Val Loss: 1.0201
Epoch 100/100| Train Loss: 0.7656| Val Loss: 1.0201
----- Mode:  mean --------
Epoch 10/100| Train Loss: 0.8482| Val Loss: 0.8887
Epoch 20/100| Train Loss: 0.8230| Val Loss: 0.9532
Epoch 30/100| Train Loss: 0.7928| Val Loss: 0.8936
Epoch 40/100| Train Loss: 0.7731| Val Loss: 0.9394
Epoch 50/100| Train Loss: 0.7590| Val Loss: 0.9128
Epoch 60/100| Train Loss: 0.7505| Val Loss: 0.8694
Epoch 70/100| Train Loss: 0.7475| Val Loss: 0.8842
Epoch 80/100| Train Loss: 0.7446| Val Loss: 0.8899
Epoch 90/100| Train Lo

In [9]:
losses.keys()

dict_keys(['attention train', 'attention test', 'mean train', 'mean test', 'none train', 'none test'])

In [10]:
for name, loss in losses.items():
    if "test" in name:
        print("-----",name,"-----\n",loss)

----- attention test -----
 (2.845211824025863, tensor([0.6051, 1.2931, 1.3306]), tensor([[ 0.4388, -0.4275, -0.4229],
        [ 0.4158, -0.4087, -0.4145],
        [-0.3495,  0.5985,  0.6105],
        ...,
        [ 0.3463, -0.2992, -0.3261],
        [-0.3662,  0.6242,  0.0760],
        [ 0.4289, -0.4306, -0.4125]]), tensor([[ 0.9579, -0.5110, -0.5110],
        [ 0.4893, -0.5110, -0.5110],
        [-0.3829, -0.5110,  0.8185],
        ...,
        [ 0.0597,  3.9559,  3.9559],
        [-1.0338,  2.1479, -0.5110],
        [ 0.4893, -0.5110, -0.5110]]))
----- mean test -----
 (2.5624578872093786, tensor([0.5972, 1.2136, 1.2632]), tensor([[-0.2580,  0.8025,  0.0806],
        [ 0.4820, -0.4087, -0.4364],
        [-0.5888,  0.2818,  0.5776],
        ...,
        [ 0.1228, -0.1190,  0.0312],
        [ 0.2361, -0.2567, -0.1575],
        [ 0.4622, -0.4588, -0.4982]]), tensor([[-1.0338,  2.1479, -0.5110],
        [-1.0338,  1.7225,  3.9559],
        [-0.3829, -0.5110, -0.5110],
        ...,
     